# 🧠 Orbis Ethica - Model Training (LoRA Fine-Tuning)

This notebook fine-tunes **Llama-3 (8B)** on your custom ethical dataset.
It uses **Unsloth**, which makes training 2x faster and uses 70% less memory (runs on free Colab Tesla T4).

### Steps:
1.  **Upload Data:** Drag and drop your `training_data.jsonl` file to the files sidebar.
2.  **Run All:** Click 'Runtime' -> 'Run all'.
3.  **Download:** The model will be saved as `Orbis-7B.gguf` for you to download.

In [ ]:
%%capture
# 1. Install Unsloth and Dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 2. Load Base Model (Llama-3-8B-Instruct)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detection
load_in_4bit = True # 4bit quantization to fit in memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters (This is what we train)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 3. Load Your Data
from datasets import load_dataset

# Make sure you uploaded 'training_data.jsonl'
dataset = load_dataset("json", data_files="training_data.jsonl", split="train")

# Format prompt
alpaca_prompt = """
### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# 4. Train the Model
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increase this for real training (e.g. 1000)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

In [ ]:
# 5. Save Model (GGUF Format for M1 Mac)
model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
print("✅ Model saved! Download 'model/model-unsloth.Q4_K_M.gguf' from the files sidebar.")